## Lab2
Today we will build a GPT-2 model from scratch using PyTorch. Please download files from https://box.xjtlu.edu.cn/d/2811f983c8de49c5854a/

In [1]:
import torch
x = torch.randn(1000, 1000, device="mps")
y = x @ x  # Matrix multiplication (GPU-accelerated)
print(y.device)  # Should print: mps:0

mps:0


In [2]:
 # Importing necessary libraries
# Replace the old device line with this (Apple Silicon + Intel/Windows compatible)
import torch
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")  # Apple Silicon GPU
elif torch.cuda.is_available():
    device = torch.device("cuda") # NVIDIA GPU (non-Mac)
else:
    device = torch.device("cpu")  # Intel Mac/old hardware
print(f"Using device: {device}")
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
import os  # Add this import for file operations
from tqdm import tqdm

Using device: mps


### Building a character-level Tokenizer for Chinese
- For simplicity, we read all the text from the training set to build the vocabulary.

In [3]:
class SimpleTokenizer:
    def __init__(self):
        self.word2idx = {"<pad>": 0, "<unk>": 1, "<eos>": 2}
        self.idx2word = {0: "<pad>", 1: "<unk>", 2: "<eos>"}

    def build_vocab(self, texts):
        for text in texts:
            # For Chinese text, each character is a token
            for char in text:
                if char.strip() and char not in self.word2idx:
                    idx = len(self.word2idx)
                    self.word2idx[char] = idx
                    self.idx2word[idx] = char

    def encode(self, text):
        tokens = []
        # Process each character in the text
        for char in text:
            if char.strip():  # Skip empty spaces
                tokens.append(self.word2idx.get(char, self.word2idx["<unk>"]))

        return tokens + [self.word2idx["<eos>"]]

    def decode(self, tokens):
        # For Chinese, we don't need spaces between characters
        return "".join([self.idx2word.get(token, "<unk>") for token in tokens])

In [4]:
tokenizer = SimpleTokenizer()
tokenizer.build_vocab("中国是一个伟大的国家")
print(tokenizer.encode('国家'))
print(tokenizer.encode('ab'))
print(tokenizer.decode([4,5]))



[4, 11, 2]
[1, 1, 2]
国是


### Streaming Dataset
- We will implement a custom dataset class that streams data from a text file.

In [5]:
def load_text_chunk(file_path, chunk_size=1000, start_pos=0):
    """Load a chunk of text from file starting at a specific position"""
    with open(file_path, 'r', encoding='utf-8') as f:
        f.seek(start_pos)
        chunk = f.read(chunk_size)
        new_pos = f.tell()

    # Process text differently for Chinese characters
    processed_text = []
    for char in chunk:
        # if '\u4e00' <= char <= '\u9fff':  # Chinese character range
        #     processed_text.append(char)  # Add each Chinese character as a token
        # else:
        #     # For non-Chinese, keep the original word-based approach
            processed_text.append(char)

    return processed_text, new_pos

class StreamingTextDataset(Dataset):
    def __init__(self, file_path, tokenizer, seq_length, total_samples):
        self.file_path = file_path
        self.tokenizer = tokenizer
        self.seq_length = seq_length
        self.total_samples = total_samples
        self.chunk_size = seq_length * 100  # Read larger chunks for efficiency
        self.words_buffer = []
        self.current_pos = 0

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        # Ensure buffer has enough words
        while len(self.words_buffer) < self.seq_length + 1:
            chunk_words, self.current_pos = load_text_chunk(
                self.file_path,
                chunk_size=self.chunk_size,
                start_pos=self.current_pos
            )
            if not chunk_words:  # End of file
                # Reset to beginning of file if we reach the end
                self.current_pos = 0
                chunk_words, self.current_pos = load_text_chunk(
                    self.file_path,
                    chunk_size=self.chunk_size,
                    start_pos=0
                )
            self.words_buffer.extend(chunk_words)

        # Get sequence and remove used words to save memory
        sequence = ''.join(self.words_buffer[:self.seq_length + 1])
        self.words_buffer = self.words_buffer[self.seq_length // 2:]  # Sliding window with overlap

        tokens = self.tokenizer.encode(sequence)

        # Ensure fixed length by padding or truncating
        if len(tokens) > self.seq_length + 1:
            tokens = tokens[:self.seq_length + 1]
        elif len(tokens) < self.seq_length + 1:
            tokens = tokens + [self.tokenizer.word2idx["<pad>"]] * (self.seq_length + 1 - len(tokens))

        input_tensor = torch.tensor(tokens[:-1])
        target_tensor = torch.tensor(tokens[1:])

        return input_tensor, target_tensor

In [6]:
dataset = StreamingTextDataset("wiki.txt", tokenizer, seq_length=128, total_samples=100)
print(dataset[2])

(tensor([ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  5,  1,  1,  3,  4,  1,  1,  1,
         1, 10,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  5,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1]), tensor([ 1,  1,  1,  1,  1,  1,  1,  1,  1,  5,  1,  1,  3,  4,  1,  1,  1,  1,
        10,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  

## Transformer Decoder
- We will implement a simple transformer decoder block.
- The decoder will consist of a multi-head self-attention layer followed by a feed-forward neural network.
- We will use layer normalization and residual connections.
- The model will be trained to predict the next token in a sequence.
- We will use the cross-entropy loss function for training.
- We will use the Adam optimizer for training.

<img src="./attention.png" alt="Cute Cat" width="400"/>

In [7]:
class MaskedSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        batch_size, seq_length, embed_dim = x.shape

        Q = self.W_q(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        mask = torch.triu(torch.ones(seq_length, seq_length), diagonal=1).bool().to(x.device)
        scores.masked_fill_(mask, float('-inf'))

        attention_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attention_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, embed_dim)

        return self.W_o(attn_output)


class FeedForward(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-np.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)


class DecoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim):
        super().__init__()
        self.attention = MaskedSelfAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = FeedForward(embed_dim, hidden_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoding = PositionalEncoding(embed_dim)
        self.layers = nn.ModuleList([
            DecoderBlock(embed_dim, num_heads, hidden_dim) for _ in range(num_layers)
        ])
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.fc_out = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x)
        x = self.layer_norm(x)
        return self.fc_out(x)

## Training the Model


In [8]:
# Define a train model function
def train_model(model, tokenizer, dataloader, optimizer, criterion, epochs=10):
    model.train()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    for epoch in range(epochs):
        total_loss = 0
        # Add progress bar
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)
        for batch in progress_bar:
            optimizer.zero_grad()
            inputs, targets = batch
            inputs = inputs.to(device)
            targets = targets.to(device)
            # Create padding mask (1 for real tokens, 0 for padding)
            pad_mask = (inputs != 0).float()

            outputs = model(inputs)

            # Apply mask to ignore padding in loss calculation
            loss = criterion(outputs.view(-1, outputs.shape[-1]), targets.view(-1))
            loss = loss * pad_mask.view(-1)
            loss = loss.sum() / pad_mask.sum()

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            # Update progress bar with current loss
            progress_bar.set_postfix(loss=f"{loss.item():.4f}", avg_loss=f"{total_loss/(progress_bar.n+1):.4f}")

        # Print epoch summary
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(dataloader):.4f}")

        # Test text generation with multiple prompts
        prompts = ["在历史小说《三国演义》中", "本剧对《红楼梦》中丫头的故事进行了创作性的改编"]

        for prompt in prompts:
            print(f"\nPrompt: {prompt}")
            generated_text = generate_text(model, tokenizer, prompt, max_length=200)
            print(f"Generated text: {generated_text}")

In [9]:
# Gnerate text function
@torch.no_grad()
def generate_text(model, tokenizer, prompt, max_length=200, temperature=1.0, top_k=50):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    # Encode the prompt character by character
    input_ids = torch.tensor([tokenizer.encode(prompt)[:-1]])  # Remove EOS token
    input_ids = input_ids.to(device)
    generated_tokens = []
    for _ in range(max_length):
        outputs = model(input_ids)

        # Apply temperature to control randomness
        logits = outputs[:, -1, :] / temperature
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = -float('Inf')
        
        probs = F.softmax(logits, dim=-1)

        # Sample from the distribution
        next_token = torch.multinomial(probs, 1)

        input_ids = torch.cat([input_ids, next_token], dim=1)
        generated_tokens.append(next_token.item())

        if next_token.item() == tokenizer.word2idx["<eos>"]:
            break

    # Decode and return the generated text
    full_text = prompt + tokenizer.decode(generated_tokens)
    return full_text

## Start Training


In [10]:
embed_dim = 768
num_heads = 12
hidden_dim = embed_dim * 4
num_layers = 12
seq_length = 512
batch_size = 32
epochs = 5

In [ ]:
# File path
text_file_path = "wiki.txt"

print("Building vocabulary...")
tokenizer = SimpleTokenizer()
chunk_size = 10000  # Smaller chunks
file_size = os.path.getsize(text_file_path)
current_pos = 0

# Add a maximum number of chunks to process to avoid getting stuck
max_chunks_to_process = 1000

# Process the file in smaller chunks to build vocabulary
vocab_chunks_processed = 0
while current_pos < file_size and len(tokenizer.word2idx) < 20000 and vocab_chunks_processed < max_chunks_to_process:
    with open(text_file_path, 'r', encoding='utf-8') as f:
        f.seek(current_pos)
        try:
            chunk = f.read(chunk_size)
            current_pos = f.tell()
        except:
            print(f"Error reading at position {current_pos}, skipping ahead")
            current_pos += chunk_size
            continue

    # Process chunk for vocabulary
    chunk = chunk.replace('\n', ' ').replace('.', ' . ')
    tokenizer.build_vocab([chunk])

    vocab_chunks_processed += 1
    if vocab_chunks_processed % 10 == 0:  # Report more frequently
        print(f"Processed {vocab_chunks_processed} chunks, vocabulary size: {len(tokenizer.word2idx)}")

print(f"Final vocabulary size: {len(tokenizer.word2idx)}")
# Limit total samples to a reasonable number for faster training
total_samples = 10000

print("Creating dataset...")
# Create streaming dataset and dataloader
dataset = StreamingTextDataset(text_file_path, tokenizer, seq_length, total_samples)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

print("Initializing model...")
# Initialize model, optimizer, and criterion
model = TransformerDecoder(len(tokenizer.word2idx), embed_dim, num_heads, hidden_dim, num_layers)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.CrossEntropyLoss()

print("Starting training...")
# Train the model
train_model(model, tokenizer, dataloader, optimizer, criterion, epochs)

print("Saving model...")
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'tokenizer': tokenizer,
}, 'gpt2.pt')


Building vocabulary...
Processed 10 chunks, vocabulary size: 3325
Processed 20 chunks, vocabulary size: 3963
Processed 30 chunks, vocabulary size: 4362
Processed 40 chunks, vocabulary size: 4607
Processed 50 chunks, vocabulary size: 4879
Processed 60 chunks, vocabulary size: 5117
Processed 70 chunks, vocabulary size: 5329
Processed 80 chunks, vocabulary size: 5483
Processed 90 chunks, vocabulary size: 5602
Processed 100 chunks, vocabulary size: 5718
Processed 110 chunks, vocabulary size: 5824
Processed 120 chunks, vocabulary size: 5937
Processed 130 chunks, vocabulary size: 6054
Processed 140 chunks, vocabulary size: 6156
Processed 150 chunks, vocabulary size: 6326
Processed 160 chunks, vocabulary size: 6411
Processed 170 chunks, vocabulary size: 6474
Processed 180 chunks, vocabulary size: 6702
Processed 190 chunks, vocabulary size: 6822
Processed 200 chunks, vocabulary size: 6908
Processed 210 chunks, vocabulary size: 6991
Processed 220 chunks, vocabulary size: 7028
Processed 230 chun

Epoch 1/5: 100%|█| 313/313 [2:20:12<00:00, 26.88s/it, avg_loss=6.1057,


Epoch 1/5, Loss: 6.1057

Prompt: 在历史小说《三国演义》中
Generated text: 在历史小说《三国演义》中心：《B.IAantamesitlzysitotgh，包括Nerardat.eatieiséfemsengicnteontCatinlmeeiatafLenace），以在193699995AnetontetOIdeldesirebABMarofectuladiAlestesidomeeACAleponmeuseengALeectethoBanarchaceFlyyalonteGiCohenDyce

Prompt: 本剧对《红楼梦》中丫头的故事进行了创作性的改编
Generated text: 本剧对《红楼梦》中丫头的故事进行了创作性的改编用。简力的大学会的球会被选布马人。19日，包括为是《西县*马和东县。*Pazeiciaclirtaiemomeapsseh，在阿德里斯尼西亚省机场*LFaupersitstames.Rtalateshesihes）。*Bemekachtigomn<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>


Epoch 2/5: 100%|█| 313/313 [13:16:56<00:00, 152.77s/it, avg_loss=5.134


Epoch 2/5, Loss: 5.1343

Prompt: 在历史小说《三国演义》中
Generated text: 在历史小说《三国演义》中国王为是国王王室《南》、《南文文学家》（《天发作乱》，不过》）。十年在当年间，父，是四》将自行为《高达在中心（》），孙之，李肇》（《五年》）《黄王元之后《明（“《中国元十年，被拓传《王子，“书），李、《十一百年，李王之子王，王，孙忠刘王王《国王王，后，被唐代宗仪李宗说《南氏，宗守与王宗国皇子为宗王氏氏、唐氏，成为不复（18。李氏，年）和《四五王朝官*宗氏》（《太。《百李宗元》*杨宗*王子（《关三王

Prompt: 本剧对《红楼梦》中丫头的故事进行了创作性的改编
Generated text: 本剧对《红楼梦》中丫头的故事进行了创作性的改编》，有了相当时期间的单位。200001年6年8号日本年1月，并担任国的东西岸。日至安在15月5年，他的中国人开发行了一郎的一年1198月8月，曾担任务长期出生活出自己民主办事记载，被称为在西岸营，因，而自称为他也是在了他们。大陆续出现，即从当地，每年194日本的同时的一个。他对在台中华人员，在同时他参见后，以及北部进行了多次，并入侵后来的大将打者，他在法律宾的问题中国家，在11年轻，他们会给民族人


Epoch 3/5:   0%| | 1/313 [00:13<1:10:11, 13.50s/it, avg_loss=4.6997, l

### Generate samples
- to support future generation, load weights from saved weights

In [ ]:
checkpoint = torch.load('gpt2.pt', map_location='cpu',weights_only=False)
tokenizer = checkpoint['tokenizer']
model = TransformerDecoder(len(tokenizer.word2idx), embed_dim, num_heads, hidden_dim, num_layers)

model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
prompt = "在历史小说《三国演义》中"
generated_text = generate_text(model, tokenizer, prompt, max_length=100, top_k=100)
print(f"Generated text: {generated_text}")

### Tasks
- Can you generate text with a different prompt?
- Can you modify the model to use a different tokenizer? e.g. BertTokenizer
- Can you train the model on the Tang Poetry dataset?
- Can you load a pre-trained model and fine-tune it on your dataset?
- Can you build a GPT2 using HuggingFace Transformers?